In [1]:
!git clone https://github.com/finterstellar/quant_recipe.git

Cloning into 'quant_recipe'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 10 (delta 1), reused 0 (delta 0), pack-reused 0
Unpacking objects: 100% (10/10), done.


In [2]:
!ls

quant_recipe  sample_data


In [3]:
from quant_recipe.finterstellar import Common
import pandas_datareader.data as web
import pandas as pd

/usr/local/lib/python3.6/dist-packages/pandas_datareader/compat/__init__.py:7: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  from pandas.util.testing import assert_frame_equal


In [4]:
end_date = pd.Timestamp.now().date()
start_date = Common.months_before(end_date, 12)
print(start_date, end_date)

2019-09-25 2020-09-25


In [5]:
# US10Y, US10Y-3M, Initial claim, Fed Assets, PMI, Inflation
cd = ['DGS10', 'T10Y3M', 'ICSA', 'WALCL', 'INDPRO', 'T10YIE', 'SP500']

In [6]:
tickers = ''
for c in cd:
    tickers += '{}|'.format(c)

In [7]:
df = web.DataReader(cd, 'fred', start=start_date)
df

,DGS10,T10Y3M,ICSA,WALCL,INDPRO,T10YIE,SP500
DATE,,,,,,,
2019-09-25,1.73,-0.16,NaN,3857715.0,NaN,1.60,2984.87
2019-09-26,1.70,-0.13,NaN,NaN,NaN,1.57,2977.62
2019-09-27,1.69,-0.11,NaN,NaN,NaN,1.53,2961.79
2019-09-28,NaN,NaN,218000.0,NaN,NaN,NaN,NaN
2019-09-30,1.68,-0.20,NaN,NaN,NaN,1.53,2976.74
...,...,...,...,...,...,...,...
2020-09-19,NaN,NaN,870000.0,NaN,NaN,NaN,NaN
2020-09-21,0.68,0.58,NaN,NaN,NaN,1.62,3281.06
2020-09-22,0.68,0.58,NaN,NaN,NaN,1.62,3315.57


In [20]:
set_ = []
for c in cd:
    each_ = df[c]
    each_ = each_.dropna().pct_change()
    each_
    set_.append(each_)

In [26]:
pd.options.mode.use_inf_as_na = True
daily_return = set_[0]
for i in range(1, len(cd)):
    daily_return = pd.concat([daily_return, set_[i]], axis=1)
daily_return.fillna(method='ffill', inplace=True)
daily_return.dropna(inplace=True)
daily_return

,DGS10,T10Y3M,ICSA,WALCL,INDPRO,T10YIE,SP500
DATE,,,,,,,
2019-11-01,0.023669,0.400000,0.018779,0.012882,0.009280,0.032468,0.009662
2019-11-02,0.023669,0.400000,-0.023041,0.012882,0.009280,0.032468,0.009662
2019-11-04,0.034682,0.238095,-0.023041,0.012882,0.009280,0.031447,0.003704
2019-11-05,0.039106,0.153846,-0.023041,0.012882,0.009280,0.024390,-0.001186
2019-11-06,-0.026882,-0.166667,-0.023041,0.004881,0.009280,-0.011905,0.000703
...,...,...,...,...,...,...,...
2020-09-19,0.014493,0.000000,0.004619,0.007683,0.003648,0.006024,-0.011183
2020-09-21,-0.028571,-0.033333,0.004619,0.007683,0.003648,-0.029940,-0.011571
2020-09-22,0.000000,0.000000,0.004619,0.007683,0.003648,0.000000,0.010518


In [72]:
standardize = lambda x: (x-x.mean()) / x.std()
standardized_return = standardize(daily_return)
standardized_return.head()

,DGS10,T10Y3M,ICSA,WALCL,INDPRO,T10YIE,SP500
DATE,,,,,,,
2019-11-01,0.394916,0.863987,-0.129153,0.009635,0.310304,0.507378,0.421158
2019-11-02,0.394916,0.863987,-0.155669,0.009635,0.310304,0.507378,0.421158
2019-11-04,0.556388,0.539321,-0.155669,0.009635,0.310304,0.489890,0.146020
2019-11-05,0.621250,0.370378,-0.155669,0.009635,0.310304,0.369025,-0.079777
2019-11-06,-0.346220,-0.272343,-0.155669,-0.280689,0.310304,-0.252660,0.007418


In [73]:
standardized_return = standardized_return[-90:]
standardized_return.head()

,DGS10,T10Y3M,ICSA,WALCL,INDPRO,T10YIE,SP500
DATE,,,,,,,
2020-06-10,-1.522955,-0.154082,-0.251691,-0.438965,1.429971,0.087195,-0.270366
2020-06-11,-1.711458,-0.249294,-0.251691,-0.438965,1.429971,-0.857976,-2.746901
2020-06-12,1.158609,0.307417,-0.251691,-0.438965,1.429971,-0.048747,0.578092
2020-06-13,1.158609,0.307417,-0.151586,-0.438965,1.429971,-0.048747,0.578092
2020-06-15,0.047902,-0.011048,-0.151586,-0.438965,1.429971,0.234371,0.358812


In [74]:
factors = standardized_return.iloc[:,:-1].copy()
factors.tail(3)

,DGS10,T10Y3M,ICSA,WALCL,INDPRO,T10YIE
DATE,,,,,,
2020-09-22,0.047902,0.061872,-0.138131,-0.179013,0.189358,-0.048747
2020-09-23,0.047902,0.027298,-0.138131,-0.310452,0.189358,-0.260212
2020-09-24,0.047902,0.061872,-0.138131,-0.310452,0.189358,-0.262856


In [75]:
indice = standardized_return.iloc[:,-1].copy()
indice.tail(3)

DATE
2020-09-22    0.460667
2020-09-23   -1.120413
2020-09-24    0.112927
Name: SP500, dtype: float64

In [76]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt 
import numpy as np

pca = PCA(n_components=factors.shape[1])
pca.fit(factors)
np.cumsum(pca.explained_variance_ratio_)

array([0.58107053, 0.88093027, 0.9531116 , 0.99466425, 0.99788077,
       1.        ])

In [78]:
pca = PCA(n_components=1)
pca.fit(factors)

# Coefficients
coef = pd.DataFrame(pca.components_.T, columns=['pc'], index = factors.columns)
abs(coef).rank(ascending=False)

# US10Y, US10Y-3M, Initial claim, Fed Assets, PMI, Inflation

,pc
DGS10,1.0
T10Y3M,4.0
ICSA,6.0
WALCL,5.0
INDPRO,3.0
T10YIE,2.0


In [49]:
pc = pca.fit_transform(factors)[:, 0]

standardized_return['pc'] = pc
abs(standardized_return.corr().iloc[-1])

DGS10     0.170570
T10Y3M    0.010246
ICSA      0.735862
WALCL     0.905711
INDPRO    0.684236
T10YIE    0.385297
SP500     0.031519
pc        1.000000
Name: pc, dtype: float64

In [79]:
X = factors[['DGS10', 'T10YIE', 'INDPRO']]

In [80]:
from sklearn import linear_model

linear_regression = linear_model.LinearRegression()
linear_regression.fit(X = X, y = indice)
prediction = linear_regression.predict(X = X)
print('a value = ', linear_regression.intercept_)
print('b value = ', linear_regression.coef_)

a value =  -0.013775963427089213
b value =  [ 0.16200054  0.797087   -0.0536442 ]


In [81]:
# Initial claim, Fed Assets, PMI, Inflation

In [82]:
from sklearn.linear_model import LinearRegression

regr = LinearRegression().fit(X, indice)

In [83]:
regr.coef_

array([ 0.16200054,  0.797087  , -0.0536442 ])

In [84]:
regr.intercept_

-0.013775963427089213

In [85]:
regr.score(X, indice)

0.297261420016096

,ICSA,WALCL,INDPRO,T10YIE
DATE,,,,
2019-11-01,-0.129153,0.009635,0.310304,0.507378
2019-11-02,-0.155669,0.009635,0.310304,0.507378
2019-11-04,-0.155669,0.009635,0.310304,0.489890
2019-11-05,-0.155669,0.009635,0.310304,0.369025
2019-11-06,-0.155669,-0.280689,0.310304,-0.252660
...,...,...,...,...
2020-09-19,-0.138131,-0.179013,0.189358,0.054437
2020-09-21,-0.138131,-0.179013,0.189358,-0.561582
2020-09-22,-0.138131,-0.179013,0.189358,-0.048747
